# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ni7chal14/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:

import pandas as pd
import numpy as np

# Load the starter dataset.
# Adjust the path if your notebook already loads df in an earlier cell.
from pathlib import Path

possible_paths = [
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("/content/content_refresh_anonymized (1).csv"),
]

data_path = next((p for p in possible_paths if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Could not find content_refresh_anonymized.csv. "
        "Check the dataset path."
    )

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("Unique clients:", df["client_id"].nunique())

# Exploratory outcome only: NOT a scoring feature.
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

# SIGNAL 1: Freshness vs observed decline
freshness_order = ["0-30", "31-90", "91-180", "181+"]

freshness_table = (
    df.groupby("freshness_tier", observed=False)
      .agg(
          n=("content_id", "size"),
          decline_rate=("is_declining_label", "mean"),
          median_impressions=("impressions_90d", "median")
      )
      .reindex(freshness_order)
)

freshness_table["decline_rate_pct"] = (
    freshness_table["decline_rate"] * 100
).round(1)

print("\nSIGNAL 1: Freshness vs observed decline")
display(
    freshness_table[
        ["n", "decline_rate_pct", "median_impressions"]
    ].reset_index()
)

# SIGNAL 2: CTR vs position.
# avg_position == 0 means no position data, so exclude it.
position_df = df.loc[df["avg_position"] > 0].copy()

position_df["position_bucket"] = pd.cut(
    position_df["avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"],
    include_lowest=True
)

position_table = (
    position_df.groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_ctr=("ctr", "median"),
        median_impressions=("impressions_90d", "median")
    )
    .reset_index()
)

print("\nSIGNAL 2: CTR vs search position")
display(position_table)

print("\nInitial verdicts:")
print("Freshness vs decline: MIXED")
print("CTR vs position: MIXED")

Dataset shape: (30000, 44)
Unique clients: 32

SIGNAL 1: Freshness vs observed decline


,freshness_tier,n,decline_rate_pct,median_impressions
0,0-30,20480,51.1,470.0
1,31-90,175,58.9,510.0
2,91-180,9171,61.1,1692.0
3,181+,174,47.1,15.5



SIGNAL 2: CTR vs search position


,position_bucket,n,median_ctr,median_impressions
0,1-3,1141,0.00,74.0
1,4-10,11842,0.16,1184.0
2,11-20,7273,0.10,870.0
3,21-50,7225,0.03,807.0
4,50+,1314,0.00,219.5



Initial verdicts:
Freshness vs decline: MIXED
CTR vs position: MIXED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:

# Transparent, rule-based baseline. No fitted weights.

df["stale_flag"] = (
    df["days_since_last_update"] >= 91
).astype(int)

df["visible_flag"] = (
    df["impressions_90d"] >= 300
).astype(int)

df["baseline_score"] = (
    df["stale_flag"]
    * df["visible_flag"]
    * df["impressions_90d"]
)

eligible = (
    (df["stale_flag"] == 1)
    & (df["visible_flag"] == 1)
)

df["reason_code"] = np.where(
    eligible,
    "stale_but_visible",
    "not_eligible"
)

df["action"] = np.where(
    eligible,
    "review_for_refresh",
    "no_refresh_priority"
)

# Rank the queue from highest to lowest score.
ranked = df.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).copy()

ranked.insert(
    0,
    "rank",
    range(1, len(ranked) + 1)
)

# Export the required output.
output_dir = Path("work/outputs")
if not output_dir.exists():
    output_dir = Path("../../work/outputs")

output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

output_columns = [
    "rank",
    "content_id",
    "client_id",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "baseline_score",
    "reason_code",
    "action"
]

ranked[output_columns].to_csv(output_path, index=False)

print("Ranked queue saved to:", output_path)
print("Total content items:", len(ranked))
print("Refresh-review candidates:", int(eligible.sum()))
print("Non-eligible items:", int((~eligible).sum()))

display(ranked[output_columns].head(10))

Ranked queue saved to: ../../work/outputs/baseline_action_score.csv
Total content items: 30000
Refresh-review candidates: 7234
Non-eligible items: 22766


,rank,content_id,client_id,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,baseline_score,reason_code,action
6653,1,content_5fe46e04994d,client_4e07408562,104,517715,741,0.14,4.2,517715,stale_but_visible,review_for_refresh
29400,2,content_2dba2b1f9536,client_6208ef0f77,104,443434,910,0.21,27.9,443434,stale_but_visible,review_for_refresh
13537,3,content_2c2606c5d176,client_19581e27de,104,347399,1854,0.53,4.2,347399,stale_but_visible,review_for_refresh
26531,4,content_cb112fce36be,client_19581e27de,104,309910,492,0.16,5.6,309910,stale_but_visible,review_for_refresh
21565,5,content_9532f197bbc8,client_4e07408562,104,309192,2689,0.87,2.0,309192,stale_but_visible,review_for_refresh
3394,6,content_36ff89c8214e,client_19581e27de,104,295097,154,0.05,7.3,295097,stale_but_visible,review_for_refresh
26798,7,content_b28d1efd668f,client_6208ef0f77,104,286608,169,0.06,26.2,286608,stale_but_visible,review_for_refresh
23767,8,content_813e88069237,client_6208ef0f77,104,233561,129,0.06,26.2,233561,stale_but_visible,review_for_refresh
26255,9,content_c21024970297,client_19581e27de,104,211366,870,0.41,5.1,211366,stale_but_visible,review_for_refresh
7445,10,content_c8e9d6ab9013,client_19581e27de,104,208678,0,0.00,9.7,208678,stale_but_visible,review_for_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:

# Review the 20 highest-ranked candidates.
review = ranked.loc[ranked["baseline_score"] > 0].head(20).copy()

def explain_candidate(row):
    age = int(row["days_since_last_update"])
    impressions = int(row["impressions_90d"])
    ctr = float(row["ctr"])
    position = float(row["avg_position"])

    why = (
        f"Not updated for {age} days and received "
        f"{impressions:,} impressions in the 90-day window."
    )

    if position > 0 and position <= 10:
        why += (
            f" Average position is {position:.1f}, "
            "so it already has first-page visibility."
        )
    elif position > 10:
        why += (
            f" Average position is {position:.1f}; "
            "ranking improvement may be worth investigating."
        )
    else:
        why += " Average position is unavailable."

    if ctr < 0.1:
        why += f" CTR is low ({ctr:.2f}%)."
    else:
        why += f" CTR is {ctr:.2f}%."

    return why

def challenge_candidate(row):
    position = float(row["avg_position"])
    ctr = float(row["ctr"])
    clicks = int(row["clicks_90d"])

    if clicks == 0:
        return (
            "Verify tracking and query relevance; impressions without "
            "clicks may indicate poor intent match or measurement issues."
        )
    if position > 50:
        return (
            "Check whether the page targets relevant queries and whether "
            "a refresh can realistically improve its very weak visibility."
        )
    if position > 0 and position <= 3 and ctr < 0.1:
        return (
            "Check query mix, SERP features, and CTR measurement; the page "
            "already ranks highly, so a content refresh may not be the fix."
        )
    if position > 0 and position <= 10:
        return (
            "Check whether the content is outdated or inaccurate; strong "
            "rankings may mean a refresh is unnecessary or risky."
        )
    return (
        "Check search intent, content quality, and whether impressions "
        "represent valuable queries before prioritizing a refresh."
    )

review["review_action"] = "Review for possible refresh"
review["why_selected"] = review.apply(explain_candidate, axis=1)
review["what_could_make_it_wrong"] = review.apply(
    challenge_candidate, axis=1
)

review_columns = [
    "rank",
    "content_id",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "reason_code",
    "review_action",
    "why_selected",
    "what_could_make_it_wrong",
]

display(review[review_columns])

assert len(review) == min(20, int((ranked["baseline_score"] > 0).sum()))
assert review["why_selected"].notna().all()
assert review["what_could_make_it_wrong"].notna().all()

print(f"Reviewed {len(review)} candidates.")

,rank,content_id,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,reason_code,review_action,why_selected,what_could_make_it_wrong
6653,1,content_5fe46e04994d,104,517715,741,0.14,4.2,stale_but_visible,Review for possible refresh,"Not updated for 104 days and received 517,715 ...",Check whether the content is outdated or inacc...
29400,2,content_2dba2b1f9536,104,443434,910,0.21,27.9,stale_but_visible,Review for possible refresh,"Not updated for 104 days and received 443,434 ...","Check search intent, content quality, and whet..."
13537,3,content_2c2606c5d176,104,347399,1854,0.53,4.2,stale_but_visible,Review for possible refresh,"Not updated for 104 days and received 347,399 ...",Check whether the content is outdated or inacc...
26531,4,content_cb112fce36be,104,309910,492,0.16,5.6,stale_but_visible,Review for possible refresh,"Not updated for 104 days and received 309,910 ...",Check whether the content is outdated or inacc...
21565,5,content_9532f197bbc8,104,309192,2689,0.87,2.0,stale_but_visible,Review for possible refresh,"Not updated for 104 days and received 309,192 ...",Check whether the content is outdated or inacc...
3394,6,content_36ff89c8214e,104,295097,154,0.05,7.3,stale_but_visible,Review for possible refresh,"Not updated for 104 days and received 295,097 ...",Check whether the content is outdated or inacc...
26798,7,content_b28d1efd668f,104,286608,169,0.06,26.2,stale_but_visible,Review for possible refresh,"Not updated for 104 days and received 286,608 ...","Check search intent, content quality, and whet..."
23767,8,content_813e88069237,104,233561,129,0.06,26.2,stale_but_visible,Review for possible refresh,"Not updated for 104 days and received 233,561 ...","Check search intent, content quality, and whet..."
26255,9,content_c21024970297,104,211366,870,0.41,5.1,stale_but_visible,Review for possible refresh,"Not updated for 104 days and received 211,366 ...",Check whether the content is outdated or inacc...
7445,10,content_c8e9d6ab9013,104,208678,0,0.00,9.7,stale_but_visible,Review for possible refresh,"Not updated for 104 days and received 208,678 ...",Verify tracking and query relevance; impressio...


Reviewed 20 candidates.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:

# Identify potentially weak recommendations in the top-20 queue.
weak_picks = review.loc[
    (review["clicks_90d"] == 0)
    | (review["avg_position"] > 50)
    | (
        (review["avg_position"] > 0)
        & (review["avg_position"] <= 3)
        & (review["ctr"] < 0.1)
    )
]

print("Potentially weak picks:", len(weak_picks))

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "avg_position",
            "what_could_make_it_wrong",
        ]
    ]
)

# Leakage guard: these fields must never enter the score formula.
forbidden_features = {
    "trend_pct",
    "trend_direction",
    "is_declining_label",
    "content_id",
    "client_id",
}

actual_score_inputs = {
    "days_since_last_update",
    "impressions_90d",
}

assert actual_score_inputs.isdisjoint(forbidden_features)
assert "trend_pct" not in actual_score_inputs
assert "trend_direction" not in actual_score_inputs
assert "is_declining_label" not in actual_score_inputs

# Confirm that the score is exactly the stated transparent rule.
expected_score = (
    (df["days_since_last_update"] >= 91).astype(int)
    * (df["impressions_90d"] >= 300).astype(int)
    * df["impressions_90d"]
)

assert np.allclose(
    df["baseline_score"].to_numpy(),
    expected_score.to_numpy()
)

print("Leakage check passed.")
print("Score uses only freshness and 90-day impressions.")
print("Trend fields and identifiers are not score inputs.")

Potentially weak picks: 1


,rank,content_id,impressions_90d,clicks_90d,ctr,avg_position,what_could_make_it_wrong
7445,10,content_c8e9d6ab9013,208678,0,0.0,9.7,Verify tracking and query relevance; impressio...


Leakage check passed.
Score uses only freshness and 90-day impressions.
Trend fields and identifiers are not score inputs.


### Baseline rule

The baseline prioritizes content that has not been updated for at least 91 days and has received at least 300 impressions in the trailing 90-day window. Eligible content receives a score equal to its 90-day impressions; all other content receives a score of zero.

### Reason code and action

- **Reason code:** `stale_but_visible`
- **Action:** `review_for_refresh`

The queue is intended to support human review, not automatically trigger content changes. The thresholds are transparent operational choices, not learned or optimized weights.

### Limitations

The score rewards visibility and freshness age, but it does not establish that a page is declining or that refreshing it will improve performance. High impressions may reflect existing success, while low CTR or poor average position may have causes unrelated to content freshness. The queue should therefore be treated as a prioritization heuristic.

### Leakage check

The score does not use `trend_pct`, `trend_direction`, or `is_declining_label`, which are related to the target definition. Pseudonymous content and client IDs are retained only for identification. The decline label is used for exploratory signal validation, not for constructing the score.

### Weak-pick review

Potential weak picks include pages with impressions but zero clicks, pages ranking beyond position 50, and pages already in positions 1–3 with very low CTR. These cases need further investigation before a refresh is recommended. Each reviewed candidate has a documented reason for selection and a condition that could make the recommendation wrong.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.